# Root finding

All four methods find x such that f(x) = 0 on an interval [a, b].

| Method | Convergence | Needs f' | Needs bracket |
|---|---|---|---|
| Bisection | Linear (p=1) | No | Yes |
| Newton-Raphson | Quadratic (p=2) | Yes | No |
| Secant | Superlinear (p≈1.618) | No | No |
| Brent | Superlinear, guaranteed | No | Yes |

In [ ]:
import sys
sys.path.insert(0, '..')

import math
import matplotlib.pyplot as plt
import numpy as np

from src.root_finding import bisection, newton, secant, brent

## Bisection

The simplest bracketing method. At each step, evaluate f at the midpoint of [a, b] and discard the half that does not contain the root. Error halves each iteration, so convergence is linear with p = 1.

Error after n steps: |e_n| ≤ (b - a) / 2^n

In [ ]:
f = lambda x: x**2 - 2  # root at sqrt(2)
root, iters, errors = bisection(f, 1.0, 2.0, tol=1e-12)
print(f'root={root:.15f}, iterations={iters}')
print(f'|root - sqrt(2)| = {abs(root - math.sqrt(2)):.2e}')

plt.semilogy(errors, marker='o', markersize=3)
plt.xlabel('iteration')
plt.ylabel('|error|')
plt.title('Bisection convergence on x^2 - 2')
plt.tight_layout()
plt.show()

## Newton-Raphson

Uses the tangent line at x_n to compute the next iterate:

    x_{n+1} = x_n - f(x_n) / f'(x_n)

Converges quadratically near a simple root: |e_{n+1}| ≈ (f''(x*) / 2f'(x*)) |e_n|^2

In [ ]:
f  = lambda x: x**2 - 2
df = lambda x: 2*x

root, iters, errors = newton(f, df, x0=1.5, tol=1e-12)
print(f'root={root:.15f}, iterations={iters}')
print(f'|root - sqrt(2)| = {abs(root - math.sqrt(2)):.2e}')

plt.semilogy(errors, marker='o', markersize=4)
plt.xlabel('iteration')
plt.ylabel('|error|')
plt.title('Newton convergence on x^2 - 2')
plt.tight_layout()
plt.show()

## Secant method

Approximates f' using the previous two iterates:

    x_{n+1} = x_n - f(x_n) * (x_n - x_{n-1}) / (f(x_n) - f(x_{n-1}))

No derivative required. Convergence order p ≈ 1.618 (golden ratio).

In [ ]:
f = lambda x: x**2 - 2
root, iters, errors = secant(f, x0=1.0, x1=2.0, tol=1e-12)
print(f'root={root:.15f}, iterations={iters}')
print(f'|root - sqrt(2)| = {abs(root - math.sqrt(2)):.2e}')

## Brent's method

Combines bisection (guaranteed to converge) with inverse quadratic interpolation and the secant method (fast when the function is smooth). Falls back to bisection whenever the faster step would go outside the bracket.

This is what scipy.optimize.brentq uses internally.

In [ ]:
import scipy.optimize

f = lambda x: math.cos(x) - x  # root near 0.739085

our_root, iters, _ = brent(f, 0.0, 1.0, tol=1e-12)
scipy_root = scipy.optimize.brentq(f, 0.0, 1.0, xtol=1e-12)

print(f'our root:   {our_root:.15f}  ({iters} iters)')
print(f'scipy root: {scipy_root:.15f}')
print(f'difference: {abs(our_root - scipy_root):.2e}')

## Convergence comparison

All four methods applied to x^2 - 2 = 0. Newton wins on iteration count; bisection is slowest but never fails.

In [ ]:
f  = lambda x: x**2 - 2
df = lambda x: 2*x

_, _, e_bis  = bisection(f, 1.0, 2.0, tol=1e-14)
_, _, e_newt = newton(f, df, x0=1.5, tol=1e-14)
_, _, e_sec  = secant(f, x0=1.0, x1=2.0, tol=1e-14)
_, _, e_brt  = brent(f, 1.0, 2.0, tol=1e-14)

fig, ax = plt.subplots()
for label, errors in [('bisection', e_bis), ('newton', e_newt), ('secant', e_sec), ('brent', e_brt)]:
    ax.semilogy(errors, label=label)
ax.set_xlabel('iteration')
ax.set_ylabel('|error|')
ax.set_title('Root-finding convergence on x^2 - 2')
ax.legend()
plt.tight_layout()
plt.show()